# Actividad 2 — Modelo de probabilidad de pago

## Bloque 1: relación de cada variable con la variable objetivo

Antes de formular el modelo, mido qué tan asociada está cada variable predictora con el
pago total (`pagada_total`). Como la objetivo es categórica (sí/no), para las variables
categóricas mido la **tasa de pago por grupo**, y para las numéricas comparo sus valores
entre las que pagan y las que no. Esto orienta qué variables tienen señal.

In [2]:
import duckdb
import pandas as pd

# Leo la sábana desde la base analítica que construyó el pipeline.
# Trabajo sobre la capa analítica, no sobre la fuente cruda.
con = duckdb.connect("/workspaces/prueba-tecnica-cxc/data/analitica.duckdb")
df = con.execute("SELECT * FROM sabana_cxc").df()

print("Sábana cargada:", df.shape[0], "obligaciones,", df.shape[1], "columnas")
print("\nDistribución de la variable objetivo (pagada_total):")
print(df["pagada_total"].value_counts())
print("\nTasa de pago general:", round(df["pagada_total"].mean() * 100, 2), "%")

Sábana cargada: 21739 obligaciones, 20 columnas

Distribución de la variable objetivo (pagada_total):
pagada_total
1    17323
0     4416
Name: count, dtype: int64

Tasa de pago general: 79.69 %


### 1.1 Variables categóricas: tasa de pago por grupo

Para cada tipo de transacción, calculo qué porcentaje se paga completo. Si la tasa varía
mucho entre tipos, esa variable tiene señal para el modelo.

In [3]:
# Tasa de pago por tipo de transacción. Solo tipos con volumen suficiente (>=100)
# para que la tasa sea confiable. Ordeno de peor a mejor recuperación.
tasa_por_trn = con.execute("""
    SELECT
        descri_cod_trn AS tipo_transaccion,
        COUNT(*) AS obligaciones,
        ROUND(AVG(pagada_total) * 100, 1) AS tasa_pago_pct
    FROM sabana_cxc
    GROUP BY 1
    HAVING COUNT(*) >= 100
    ORDER BY tasa_pago_pct ASC
""").df()
print("Tasa de pago por tipo de transacción (peor a mejor):")
print(tasa_por_trn.to_string(index=False))

Tasa de pago por tipo de transacción (peor a mejor):
                tipo_transaccion  obligaciones  tasa_pago_pct
      TRANSFERENCIA CANAL FISICO           139           36.7
   COMISION RETIRO INTERNACIONAL           127           54.3
         COMISION CONSULTA SALDO           802           67.7
COMISION TRANSFERENCIA EXTERNA B          1184           69.3
       TRANSFERENCIA BANCA MOVIL           110           70.9
         COMISION RETIRO CANAL A           645           71.6
              CARGO FISCAL IVA A           109           71.6
      APERTURA INVERSION VIRTUAL           111           73.0
  COMISION RETIRO CORRESPONSAL B          1129           73.9
     CARGO FISCAL IVA COMISION B           659           77.2
      CARGO FISCAL TRANSACCIONAL          6823           77.9
       TRANSFERENCIA BILLETERA A           507           80.7
       COBRO SERVICIO TRANSPORTE          3704           82.6
       CARGO FISCAL IVA TRASLADO          1070           83.3
     TRANSFERENCI

In [4]:
# Tasa de pago por producto y por banda de antigüedad.
# Quiero ver si el producto o la edad de la obligación mueven el pago.
tasa_producto = con.execute("""
    SELECT descri_cod_apli_prod AS producto,
           COUNT(*) AS obligaciones,
           ROUND(AVG(pagada_total) * 100, 1) AS tasa_pago_pct
    FROM sabana_cxc GROUP BY 1 ORDER BY tasa_pago_pct
""").df()

tasa_antiguedad = con.execute("""
    SELECT banda_antiguedad,
           COUNT(*) AS obligaciones,
           ROUND(AVG(pagada_total) * 100, 1) AS tasa_pago_pct
    FROM sabana_cxc GROUP BY 1 ORDER BY banda_antiguedad
""").df()

print("Tasa de pago por producto:")
print(tasa_producto.to_string(index=False))
print("\nTasa de pago por banda de antigüedad:")
print(tasa_antiguedad.to_string(index=False))

Tasa de pago por producto:
 producto  obligaciones  tasa_pago_pct
   AHORRO         21491           79.6
CORRIENTE           248           87.9

Tasa de pago por banda de antigüedad:
banda_antiguedad  obligaciones  tasa_pago_pct
           61-90            12           83.3
             91+         21727           79.7


### 1.2 Variable numérica (monto): ¿difiere entre las que pagan y las que no?

Comparo el monto original promedio y mediano entre las obligaciones que se pagan
completas y las que no. Si difieren, el monto tiene señal.

In [5]:
# Comparo el monto entre pagadas y no pagadas.
# Uso mediana además del promedio porque ya sabemos que los montos están sesgados.
monto_vs_pago = con.execute("""
    SELECT
        pagada_total,
        COUNT(*) AS obligaciones,
        ROUND(AVG(vlr_original), 2) AS monto_promedio,
        ROUND(MEDIAN(vlr_original), 2) AS monto_mediano
    FROM sabana_cxc
    GROUP BY 1
""").df()
print("Monto según se pague o no (0 = no pagada, 1 = pagada total):")
print(monto_vs_pago.to_string(index=False))

Monto según se pague o no (0 = no pagada, 1 = pagada total):
 pagada_total  obligaciones  monto_promedio  monto_mediano
            0          4416         8063.05        2473.84
            1         17323         6116.62        2136.02


### Conclusiones del Bloque 1

- Tasa de pago general: 79,69% (objetivo desbalanceada 80/20).
- Mejor predictor: tipo de transacción (tasa de pago de 36,7% a 98,1%, alto poder
  discriminante).
- Producto: discrimina poco (AHORRO 79,6% vs CORRIENTE 87,9%, este último poco robusto).
- Antigüedad: NO discrimina (99,9% en banda 91+, varianza casi nula) → se descarta.
  Consecuencia: la censura es despreciable en la práctica.
- Monto: señal débil pero coherente (no pagadas con monto mediano mayor).
- Variables candidatas: tipo de transacción, monto original, producto (con cautela).

## Bloque 2: preparación de los datos para el modelo

Aplico tres transformaciones, cada una con su justificación (ver Parte G de la bitácora):
1. **Log del monto** (G4): domar el sesgo de los montos.
2. **One-hot encoding** (G3): convertir las categorías en variables dummy numéricas,
   sin inventar jerarquías falsas.
3. **Split agrupado por titular** (G5): separar 80/20 sin que un mismo `num_cta` quede
   en entrenamiento y prueba a la vez.

Recuerdo la regla de oro: NO uso `vlr_pagado`, `vlr_pendiente_pago`, `tasa_recuperacion`
ni `estado_recuperacion` como predictores (fuga de información).

In [6]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# Selecciono SOLO las variables que decidí con evidencia en el Bloque 1.
# Dejo fuera a propósito las que causarían fuga de información.
predictoras_categoricas = ["descri_cod_trn", "descri_cod_apli_prod"]
objetivo = "pagada_total"

# 1. Transformo el monto con logaritmo (log1p maneja ceros de forma segura).
df["log_monto"] = np.log1p(df["vlr_original"])

print("Monto original vs log (primeras filas):")
print(df[["vlr_original", "log_monto"]].head())

Monto original vs log (primeras filas):
   vlr_original  log_monto
0        154.08   5.043941
1       1278.25   7.154029
2       5646.89   8.639037
3        285.77   5.658681
4      17048.76   9.743891


In [7]:
# 2. One-hot encoding de las categóricas -> variables dummy (0/1).
# drop_first=True omite una categoría de referencia (evita la 'dummy variable trap').
X_categoricas = pd.get_dummies(df[predictoras_categoricas], drop_first=True)

# Uno las dummy con el monto en log. Este es el conjunto final de predictoras.
X = pd.concat([X_categoricas, df[["log_monto"]]], axis=1)
y = df[objetivo]
grupos = df["num_cta"]

print("El modelo usará", X.shape[1], "variables predictoras.")
print("Ejemplo de nombres de columnas:", list(X.columns[:6]), "...")

El modelo usará 72 variables predictoras.
Ejemplo de nombres de columnas: ['descri_cod_trn_AJUSTE INTERES DEPOSITO', 'descri_cod_trn_APERTURA INVERSION VIRTUAL', 'descri_cod_trn_CARGO FISCAL IVA A', 'descri_cod_trn_CARGO FISCAL IVA B', 'descri_cod_trn_CARGO FISCAL IVA CANAL VIRTUAL', 'descri_cod_trn_CARGO FISCAL IVA COMISION A'] ...


### 2.1 Separación entrenamiento / prueba (agrupada por titular)

Separo 80% para entrenar y 20% para evaluar, garantizando que ningún titular esté en
ambos lados. Verifico que la separación no rompió el balance de la variable objetivo.

In [8]:
# 3. Split agrupado por titular (num_cta). Semilla fija para reproducibilidad.
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
idx_train, idx_test = next(gss.split(X, y, groups=grupos))

X_train, X_test = X.iloc[idx_train], X.iloc[idx_test]
y_train, y_test = y.iloc[idx_train], y.iloc[idx_test]

# Verifico dos cosas: tamaños, y que ningún titular esté en ambos lados.
titulares_train = set(df.iloc[idx_train]["num_cta"])
titulares_test = set(df.iloc[idx_test]["num_cta"])

print("Entrenamiento:", len(X_train), "obligaciones")
print("Prueba:", len(X_test), "obligaciones")
print("Titulares compartidos (debe ser 0):", len(titulares_train & titulares_test))
print("\nTasa de pago en entrenamiento:", round(y_train.mean()*100, 1), "%")
print("Tasa de pago en prueba:", round(y_test.mean()*100, 1), "%")

Entrenamiento: 17410 obligaciones
Prueba: 4329 obligaciones
Titulares compartidos (debe ser 0): 0

Tasa de pago en entrenamiento: 79.9 %
Tasa de pago en prueba: 78.7 %


### Conclusiones del Bloque 2

- Variables predictoras tras one-hot: muchas dummy (una por tipo de transacción) + log_monto.
- Entrenamiento 17.410 / Prueba 4.329 obligaciones.
- Titulares compartidos: 0 (split agrupado correcto).
- Tasa de pago train 79,9% vs test 78,7%: split representativo, sin sesgo.